# RealMLP on Kaggle GPU

Сначала включите GPU в настройках Kaggle Notebook.

In [ ]:
!pip install -q pytabkit==1.7.3

In [ ]:
from pathlib import Path
import json
import pickle
import shutil

import numpy as np
import pandas as pd
import polars as pl
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
data_dir = Path("/kaggle/input/competitions/playground-series-s6e10")

train_df = pl.read_csv(data_dir / "train.csv")
test_df = pl.read_csv(data_dir / "test.csv")

train_df.shape, test_df.shape

In [ ]:
binary_columns = {
    "Gender": {"Female": 0, "Male": 1},
    "Customer Type": {"disloyal Customer": 0, "Loyal Customer": 1},
    "Type of Travel": {"Personal Travel": 0, "Business travel": 1},
}

train_df = train_df.with_columns(pl.col("satisfaction").cast(pl.Int8))

for column, mapping in binary_columns.items():
    train_df = train_df.with_columns(
        pl.col(column).replace_strict(mapping, return_dtype=pl.Int8)
    )
    test_df = test_df.with_columns(
        pl.col(column).replace_strict(mapping, return_dtype=pl.Int8)
    )

train_df = train_df.with_columns(
    pl.col("Arrival Delay in Minutes").is_null().cast(pl.Int8).alias("Arrival Delay Missing")
)
test_df = test_df.with_columns(
    pl.col("Arrival Delay in Minutes").is_null().cast(pl.Int8).alias("Arrival Delay Missing")
)

In [ ]:
features = train_df.drop("id", "satisfaction").to_pandas()
test_features = test_df.drop("id").to_pandas()
target = train_df["satisfaction"].to_numpy()
train_ids = train_df["id"].to_numpy()
test_ids = test_df["id"].to_numpy()

class_categories = sorted(features["Class"].unique())
features["Class"] = pd.Categorical(features["Class"], categories=class_categories)
test_features["Class"] = pd.Categorical(test_features["Class"], categories=class_categories)

fold_numbers = np.zeros(len(features), dtype=np.int8)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_number, (_, valid_index) in enumerate(folds.split(features, target)):
    fold_numbers[valid_index] = fold_number

features.shape, test_features.shape, np.bincount(fold_numbers)

## Пробное обучение

Сначала обучаем только fold 0. После проверки времени и AUC заменим `folds_to_run` на `range(5)`.

In [ ]:
output_dir = Path("/kaggle/working/realmlp_artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

folds_to_run = [0]
n_epochs = 100

realmlp_oof = np.full(len(features), np.nan, dtype=np.float32)
test_fold_predictions = []
fold_scores = {}
fold_medians = {}

In [ ]:
for valid_fold in folds_to_run:
    print(f"\nfold {valid_fold}")

    train_mask = fold_numbers != valid_fold
    valid_mask = fold_numbers == valid_fold

    x_train = features.loc[train_mask].reset_index(drop=True)
    x_valid = features.loc[valid_mask].reset_index(drop=True)
    x_test = test_features.copy()
    y_train = target[train_mask]
    y_valid = target[valid_mask]

    arrival_median = x_train["Arrival Delay in Minutes"].median()
    fold_medians[str(valid_fold)] = float(arrival_median)

    for frame in (x_train, x_valid, x_test):
        frame["Arrival Delay in Minutes"] = (
            frame["Arrival Delay in Minutes"].fillna(arrival_median)
        )

    realmlp_model = RealMLP_TD_Classifier(
        device="cuda",
        random_state=42 + valid_fold,
        n_epochs=n_epochs,
        batch_size=1024,
        val_metric_name="cross_entropy",
        use_ls=False,
        n_cv=1,
        n_refit=0,
        verbosity=2,
    )

    realmlp_model.fit(x_train, y_train, X_val=x_valid, y_val=y_valid)

    valid_prediction = realmlp_model.predict_proba(x_valid)[:, 1]
    test_prediction = realmlp_model.predict_proba(x_test)[:, 1]
    fold_auc = roc_auc_score(y_valid, valid_prediction)

    realmlp_oof[valid_mask] = valid_prediction
    test_fold_predictions.append(test_prediction.astype(np.float32))
    fold_scores[str(valid_fold)] = float(fold_auc)

    np.save(output_dir / "realmlp_oof_partial.npy", realmlp_oof)
    np.save(output_dir / f"realmlp_test_fold_{valid_fold}.npy", test_prediction)

    realmlp_model.to("cpu")
    with open(output_dir / f"realmlp_fold_{valid_fold}.pkl", "wb") as file:
        pickle.dump(realmlp_model, file, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"fold {valid_fold} auc: {fold_auc:.9f}")

    del realmlp_model, x_train, x_valid, x_test
    torch.cuda.empty_cache()

In [ ]:
realmlp_test = np.mean(test_fold_predictions, axis=0).astype(np.float32)

np.save(output_dir / "realmlp_oof.npy", realmlp_oof)
np.save(output_dir / "realmlp_test.npy", realmlp_test)
np.save(output_dir / "fold_numbers.npy", fold_numbers)
np.save(output_dir / "train_ids.npy", train_ids)
np.save(output_dir / "test_ids.npy", test_ids)

metadata = {
    "pytabkit_version": "1.7.3",
    "n_epochs": n_epochs,
    "folds": list(folds_to_run),
    "fold_scores": fold_scores,
    "arrival_delay_medians": fold_medians,
}

with open(output_dir / "metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

archive_path = shutil.make_archive(
    "/kaggle/working/realmlp_artifacts",
    "zip",
    output_dir,
)

archive_path